# GPU Notebook - Arithmetic (Addition) Interpretability

Companion to `run_gpu.ipynb`, targeting the **addition** behaviour (carry logic).

**Key difference:** the SAEs were trained on *pooled* activations from all three
behaviours (capitals + addition + units), so they already cover arithmetic. This
notebook **reuses the existing SAE checkpoints** and therefore **skips activation
capture and SAE training entirely** - the ~50 min/prompt training cost is not
repeated here.

### Per-digit attribution (important)
Qwen tokenizes numbers **one digit at a time**, most-significant first, and the
prompt must end with a **trailing space** after `Answer:` (otherwise the model's
top prediction is just a space, and attribution has no signal). We therefore build
a **separate attribution graph for each answer digit**, teacher-forcing the
already-emitted digits into the prompt. This is what makes carry logic visible -
a single whole-answer graph only attributes to the first digit.

### What to expect
Baseline analysis shows the model is **~98% confident on every addition prompt**
(even the hardest carry cases). That means *ablation* effects will likely be small
(there is little uncertainty to remove). The **swap** experiments (carry ↔ no-carry)
at the tens digit are the most likely to produce a visible, interpretable shift.

**Runtime settings:** GPU (any tier), High-RAM recommended.


## Step 1 - Mount Drive & extract project

In [1]:
# Step 1: Mount Google Drive and extract project
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os

# Set to True if you uploaded project.zip directly to the Colab files sidebar:
uploaded_directly_to_colab = False

if uploaded_directly_to_colab:
    zip_path = "/content/project.zip"
else:
    zip_path = "/content/drive/MyDrive/mphil-project/project.zip"

if os.path.exists(zip_path):
    print(f"Extracting {zip_path}...")
    !unzip -q -o {zip_path} -d /content/
    os.chdir('/content')
    print(f"Current working directory: {os.getcwd()}")
    !ls -l
else:
    print(f"ERROR: Could not find {zip_path}.")
    print("Please upload 'project.zip' to Google Drive or the Colab files sidebar.")

Mounted at /content/drive
Extracting /content/drive/MyDrive/mphil-project/project.zip...
Current working directory: /content
total 304
drwxr-xr-x 2 root root   4096 Jul  7 23:15 configs
drwxr-xr-x 3 root root   4096 Jul  7 02:58 data
drwx------ 5 root root   4096 Jul  9 17:21 drive
-rw-r--r-- 1 root root    264 Jul  7 02:58 environment.yml
-rw-r--r-- 1 root root  19186 Jul  7 02:58 IMPLEMENTATION_PLAN.md
-rw-r--r-- 1 root root   1579 Jul  7 02:58 Instructions.md
drwxr-xr-x 3 root root   4096 Jul  9 17:21 mechanistic_data
drwxr-xr-x 3 root root   4096 Jul  7 02:58 mechanistic_dataold
drwxr-xr-x 2 root root   4096 Jul  8 15:07 mechanistic_interpretability_repro.egg-info
drwxr-xr-x 3 root root   4096 Jul  9 17:21 outputs
drwxr-xr-x 2 root root   4096 Jul  8 04:36 outputsold
-rw-r--r-- 1 root root   6551 Jul  7 02:58 project.txt
drwxr-xr-x 2 root root   4096 Jul  7 02:58 __pycache__
-rw-r--r-- 1 root root  11435 Jul  7 23:15 README.md
-rw-r--r-- 1 root root 147381 Jul  8 04:39 run_gpu.ipyn

## Step 2 - Install dependencies

In [2]:
# Step 2: Install Dependencies
!pip install --upgrade "transformers>=4.51.0" accelerate
!pip install -e .

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 140.3 MB/s eta 0:00:0000:010:01
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.1
    Uninstalling transformers-5.12.1:
      Successfully uninstalled transformers-5.12.1
Obtaining file:///content
  Preparing metadata (setup.py) ... done
  Running setup.py develop for mechanistic-interpretability-repro


## Step 3 - Generate the addition dataset
Only the arithmetic CSV is needed here.

In [3]:
# Step 3: Generate Datasets (addition CSV is what we need)
!python data/generate_datasets.py
print("\nDataset files:")
!ls -lh data/addition_data.csv

Generating Datasets...

Skipping capital sentence generation. (Pass --capitals flag if needed).
Standalone Mode complete:
Saved 1000 addition problems and 
Saved 1000 unit problems.

Dataset files:
-rw-r--r-- 1 root root 52K Jul  9 17:22 data/addition_data.csv


## Step 4 - Restore SAE checkpoints from Drive (no training)

The SAEs are behaviour-agnostic (trained on pooled activations), so we reuse the
checkpoints produced by `run_gpu.ipynb`. This cell copies them back from Drive into
the local workspace. **If they are missing**, run `run_gpu.ipynb` Step 5 first (or
uncomment the training fallback below).

In [4]:
# Step 4: Restore SAE checkpoints from Drive
import os, shutil, glob

drive_sae = '/content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints'
local_sae = '/content/mechanistic_data/sae_checkpoints'
os.makedirs(local_sae, exist_ok=True)

if os.path.isdir(drive_sae) and glob.glob(f'{drive_sae}/sae_layer*.pt'):
    for f in os.listdir(drive_sae):
        shutil.copy2(f'{drive_sae}/{f}', local_sae)
    print("Restored SAE checkpoints from Drive:")
    !ls -lh {local_sae}
else:
    print("No SAE checkpoints found in Drive at:", drive_sae)
    print("Run run_gpu.ipynb Step 5 (training) first, or uncomment the fallback below.")
    # --- Training fallback (slow): captures activations then trains all 8 layers ---
    # !python src/train.py --config configs/sae_config.yaml \
    #   --drive-dir /content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints

Restored SAE checkpoints from Drive:
total 2.0G
-rw------- 1 root root  94M Jul  8 01:07 latents_layer12.npy
-rw------- 1 root root  94M Jul  8 01:08 latents_layer16.npy
-rw------- 1 root root  94M Jul  8 01:08 latents_layer20.npy
-rw------- 1 root root  94M Jul  8 01:09 latents_layer24.npy
-rw------- 1 root root  94M Jul  8 01:10 latents_layer28.npy
-rw------- 1 root root  94M Jul  8 01:10 latents_layer32.npy
-rw------- 1 root root  94M Jul  8 01:06 latents_layer4.npy
-rw------- 1 root root  94M Jul  8 01:06 latents_layer8.npy
-rw------- 1 root root 9.5K Jul  8 01:07 sae_layer12_metadata.json
-rw------- 1 root root 161M Jul  8 01:07 sae_layer12.pt
-rw------- 1 root root 9.5K Jul  8 01:08 sae_layer16_metadata.json
-rw------- 1 root root 161M Jul  8 01:08 sae_layer16.pt
-rw------- 1 root root 9.5K Jul  8 01:08 sae_layer20_metadata.json
-rw------- 1 root root 161M Jul  8 01:08 sae_layer20.pt
-rw------- 1 root root 9.5K Jul  8 01:09 sae_layer24_metadata.json
-rw------- 1 root root 161M Ju

In [ ]:

!ls outputs

add_58_83_h_graph.html		     dallas_graph.md
add_58_83_h_graph.json		     dover_graph.html
add_58_83_h_graph.md		     dover_graph.json
add_58_83_t_graph.html		     dover_graph.md
add_58_83_t_graph.json		     inhibit_dallas_full.json
add_58_83_t_graph.md		     inhibit_kabul_full.json
add_58_83_u_graph.html		     kabul_graph.html
add_58_83_u_graph.json		     kabul_graph.json
add_58_83_u_graph.md		     kabul_graph.md
add_inhibit_58_83.json		     knockout_dallas.json
add_knockout_58_83.json		     knockout_kabul.json
add_scan_58_83.json		     scan_dallas.json
add_swap_full_nocarry_to_carry.json  scan_dover.json
add_swap_nocarry_to_carry.json	     scan_kabul.json
baselines			     swap_full_oakland_to_dallas.json
dallas_graph.html		     swap_kandahar_to_munich.json
dallas_graph.json		     swap_oakland_to_dallas.json


In [5]:
!cp /content/drive/MyDrive/mphil-project/outputs/* /content/outputs

In [33]:
!rm /content/src/attribution_graph.py
!rm /content/src/intervention.py

In [34]:
!python intervention.py --help

python3: can't open file '/content/intervention.py': [Errno 2] No such file or directory


In [40]:
# override old files in the zip
!cp /content/drive/MyDrive/mphil-project/*.py /content/src

---
## Attribution graphs - carry-arithmetic prompts (per-digit)

Qwen tokenizes numbers **one digit at a time** and emits them **most-significant
first**. So the answer `141` is three decode steps: `1` → `4` → `1`. We build a
**separate attribution graph for each digit**, teacher-forcing the already-emitted
digits into the prompt:

| Digit | Prompt (teacher-forced) | Target | What it reveals |
|-------|-------------------------|--------|-----------------|
| hundreds | `... Answer: `   | `1` | carry *propagating out* (magnitude) |
| tens     | `... Answer: 1`  | `4` | carry being *consumed* (5+8+**1**=14) |
| units    | `... Answer: 14` | `1` | carry being *generated* (8+3=11) |

**The units and tens graphs are the carry-relevant ones** - the units digit is where
the carry is born, the tens digit is where it gets added in. The hundreds digit is
just the carry propagating out, but it's still a valid "did the model resolve the
magnitude" check.

Each graph still prints a ready-to-paste `--features` string, and the downstream
intervention cells point at the **tens-digit** graph (the carry-consuming step).

**Rerun requirement:** use the dedicated tens graph cell below to rebuild
`outputs/add_58_83_t_graph.json` with `--contrast-target "3"` before running
scan, inhibition, or swap cells. Old non-contrast graph JSONs are not valid for
the carry-vs-dropped-carry intervention tests.


In [ ]:
# Graph 1: 58 + 83 = 141  (carry) -- one graph per digit
# hundreds digit: 1  (carry propagating out)
!python src/attribution_graph.py \
  --prompt "Question: What is 58 + 83? Answer: " \
  --target "1" \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/add_58_83_h_graph.json \
  --output-html outputs/add_58_83_h_graph.html \
  --output-mermaid outputs/add_58_83_h_graph.md

# tens digit: 4 vs dropped-carry 3  (carry consumed: 5+8+1=14)  <-- key carry graph
!python src/attribution_graph.py \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --target "4" \
  --contrast-target "3" \
  --layers 4 8 12 16 20 24 28 \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/add_58_83_t_graph.json \
  --output-html outputs/add_58_83_t_graph.html \
  --output-mermaid outputs/add_58_83_t_graph.md

# units digit: 1  (carry generated: 8+3=11)
!python src/attribution_graph.py \
  --prompt "Question: What is 58 + 83? Answer: 14" \
  --target "1" \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/add_58_83_u_graph.json \
  --output-html outputs/add_58_83_u_graph.html \
  --output-mermaid outputs/add_58_83_u_graph.md

In [7]:
!python src/attribution_graph.py \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --target "4" \
  --contrast-target "3" \
  --layers 4 8 12 16 20 24 28 \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/add_58_83_t_graph.json \
  --output-html outputs/add_58_83_t_graph.html \
  --output-mermaid outputs/add_58_83_t_graph.md

Loading model and tokenizer...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 177.47it/s]
Disabling gradients for all model parameters for speed and memory efficiency.
Loading SAE models from /content/mechanistic_data/sae_checkpoints to device: cuda...
Loaded SAE for layer 4 (scaling factor: 0.1519)
Loaded SAE for layer 8 (scaling factor: 0.3033)
Loaded SAE for layer 12 (scaling factor: 0.2863)
Loaded SAE for layer 16 (scaling factor: 0.3016)
Loaded SAE for layer 20 (scaling factor: 0.3414)
Loaded SAE for layer 24 (scaling factor: 0.6633)
Loaded SAE for layer 28 (scaling factor: 1.0271)
Running forward pass...

Top 5 model predictions:
  '4' (prob: 1.0000, logit: 30.62)
  '3' (prob: 0.0008, logit: 23.50)
  '2' (prob: 0.0003, logit: 22.38)
  '1' (prob: 0.0001, logit: 21.75)
  '0' (prob: 0.0001, logit: 21.50)

Target token for attribution: '4' (ID: 19, probability: 1.0000)


### Checkpoint: copy graphs to Drive

In [8]:
# Persist attribution graphs immediately (crash-safe)
import os, shutil, glob
drive_out = '/content/drive/MyDrive/mphil-project/outputs'
os.makedirs(drive_out, exist_ok=True)
for f in glob.glob('/content/outputs/add_*graph*'):
    shutil.copy2(f, drive_out)
    print("Copied", os.path.basename(f))

Copied add_58_83_h_graph.json
Copied add_58_83_h_graph.html
Copied add_58_83_u_graph.json
Copied add_58_83_u_graph.html
Copied add_58_83_u_graph.md
Copied add_58_83_t_graph.html
Copied add_58_83_t_graph.md
Copied add_58_83_h_graph.md
Copied add_58_83_t_graph.json


---
## Diagnostic 1 - Component knockouts

Run these before SAE-feature interventions. The MLP knockout tests the original SAE-feature hypothesis; attention and block knockouts test whether the decisive signal is outside the final-token MLP path.

Interpretation: MLP knockout is a localized SAE-path diagnostic; attention knockout is a component-level diagnostic; block knockout is a blunt residual-stream upper bound and is expected to be destructive.


In [9]:
# Full knockout at the TENS digit: 58 + 83, Answer: 1 -> 4 (carry-consuming step)
!python src/intervention.py \
  --mode inhibit \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --target-token "4, 3, 7" \
  --layers 4 8 12 16 20 24 28 \
  --full-knockout \
  --output outputs/add_knockout_58_83.json

Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 175.84it/s]

[1/3] Clean Model Baseline:
  - Top prediction: '4' (prob: 1.0000, logit: 30.62)
  - Target ' ': prob: 0.0000, logit: 14.31
  - Target '2': prob: 0.0003, logit: 22.38
  - Target '3': prob: 0.0008, logit: 23.50
  - Target '4': prob: 1.0000, logit: 30.62
  - Target '7': prob: 0.0000, logit: 19.50

[2/3] Running SAE Reconstruction-only Baseline (no features inhibited)...
Running forward pass with inhibition: {}...
  - Top prediction: '4' (prob: 1.0000, logit: 30.62)
  - Target ' ': prob: 0.0000, logit: 14.31
  - Target '2': prob: 0.0003, logit: 22.38
  - Target '3': prob: 0.0008, logit: 23.50
  - Target '4': prob: 1.0000, logit: 30.62
  - Target '7': prob: 0.0000, logit: 19.50

[3/3] Running Full MLP Knockout (zeroing mlp output at last token for layers [4, 8, 12, 16, 20, 24, 28])...
  - Top predict

In [10]:
# Full ATTENTION knockout at the TENS digit: tests whether attention carries the decisive signal
!python src/intervention.py \
  --mode inhibit \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --target-token "4, 3, 7" \
  --layers 4 8 12 16 20 24 28 \
  --full-knockout \
  --knockout-component attn \
  --layer-scan \
  --output outputs/add_knockout_attn_58_83.json

Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 177.27it/s]

[1/3] Clean Model Baseline:
  - Top prediction: '4' (prob: 1.0000, logit: 30.62)
  - Target ' ': prob: 0.0000, logit: 14.31
  - Target '2': prob: 0.0003, logit: 22.38
  - Target '3': prob: 0.0008, logit: 23.50
  - Target '4': prob: 1.0000, logit: 30.62
  - Target '7': prob: 0.0000, logit: 19.50

[2/3] Running SAE Reconstruction-only Baseline (no features inhibited)...
Running forward pass with inhibition: {}...
  - Top prediction: '4' (prob: 1.0000, logit: 30.62)
  - Target ' ': prob: 0.0000, logit: 14.31
  - Target '2': prob: 0.0003, logit: 22.38
  - Target '3': prob: 0.0008, logit: 23.50
  - Target '4': prob: 1.0000, logit: 30.62
  - Target '7': prob: 0.0000, logit: 19.50

[3/3] Running Full ATTN Knockout (zeroing attn output at last token for layers [4, 8, 12, 16, 20, 24, 28])...
  - Top predi

In [11]:
# Full BLOCK-output knockout at the TENS digit: very blunt upper-bound diagnostic
!python src/intervention.py \
  --mode inhibit \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --target-token "4, 3, 7" \
  --layers 4 8 12 16 20 24 28 \
  --full-knockout \
  --knockout-component block \
  --layer-scan \
  --output outputs/add_knockout_block_58_83.json

Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 177.66it/s]

[1/3] Clean Model Baseline:
  - Top prediction: '4' (prob: 1.0000, logit: 30.62)
  - Target ' ': prob: 0.0000, logit: 14.31
  - Target '2': prob: 0.0003, logit: 22.38
  - Target '3': prob: 0.0008, logit: 23.50
  - Target '4': prob: 1.0000, logit: 30.62
  - Target '7': prob: 0.0000, logit: 19.50

[2/3] Running SAE Reconstruction-only Baseline (no features inhibited)...
Running forward pass with inhibition: {}...
  - Top prediction: '4' (prob: 1.0000, logit: 30.62)
  - Target ' ': prob: 0.0000, logit: 14.31
  - Target '2': prob: 0.0003, logit: 22.38
  - Target '3': prob: 0.0008, logit: 23.50
  - Target '4': prob: 1.0000, logit: 30.62
  - Target '7': prob: 0.0000, logit: 19.50

[3/3] Running Full BLOCK Knockout (zeroing block output at last token for layers [4, 8, 12, 16, 20, 24, 28])...
  - Top pre

## Supporting diagnostic - Progressive SAE contrast-feature ablation
Ablate graph-positive features from the `4 - 3` tens-digit contrast graph. The main
quantity to watch is the gap between `4` and `3`, not just the absolute `4` logit.


In [13]:
# Scan the TENS-digit graph: 58 + 83, Answer: 1 -> 4
!python src/intervention.py \
  --mode inhibit \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --target-token "4, 3, 7" \
  --layers 4 8 12 16 20 24 28 \
  --graph-json outputs/add_58_83_t_graph.json \
  --graph-feature-sign positive \
  --scan \
  --output outputs/add_scan_58_83.json

Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 178.44it/s]
Loaded 157 nodes from graph JSON; extracted features across 7 layers (70 total features)
  Graph feature sign filter: positive (skipped 70 feature nodes)

[1/3] Clean Model Baseline:
  - Top prediction: '4' (prob: 1.0000, logit: 30.62)
  - Target ' ': prob: 0.0000, logit: 14.31
  - Target '2': prob: 0.0003, logit: 22.38
  - Target '3': prob: 0.0008, logit: 23.50
  - Target '4': prob: 1.0000, logit: 30.62
  - Target '7': prob: 0.0000, logit: 19.50

[2/3] Running SAE Reconstruction-only Baseline (no features inhibited)...
Running forward pass with inhibition: {}...
  - Top prediction: '4' (prob: 1.0000, logit: 30.62)
  - Target ' ': prob: 0.0000, logit: 14.31
  - Target '2': prob: 0.0003, logit: 22.38
  - Target '3': prob: 0.0008, logit: 23.50
  - Target '4': prob: 1.0000, logit: 30.62
  - Target '7

## Supporting diagnostic - SAE feature inhibition (weak directional effect)

In [14]:
# Inhibit all tens-digit graph features: 58 + 83, Answer: 1 -> 4
!python src/intervention.py \
  --mode inhibit \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --target-token "4, 3, 7" \
  --layers 4 8 12 16 20 24 28 \
  --graph-json outputs/add_58_83_t_graph.json \
  --graph-feature-sign positive \
  --output outputs/add_inhibit_58_83.json

Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 175.78it/s]
Loaded 157 nodes from graph JSON; extracted features across 7 layers (70 total features)
  Graph feature sign filter: positive (skipped 70 feature nodes)

[1/3] Clean Model Baseline:
  - Top prediction: '4' (prob: 1.0000, logit: 30.62)
  - Target ' ': prob: 0.0000, logit: 14.31
  - Target '2': prob: 0.0003, logit: 22.38
  - Target '3': prob: 0.0008, logit: 23.50
  - Target '4': prob: 1.0000, logit: 30.62
  - Target '7': prob: 0.0000, logit: 19.50

[2/3] Running SAE Reconstruction-only Baseline (no features inhibited)...
Running forward pass with inhibition: {}...
  - Top prediction: '4' (prob: 1.0000, logit: 30.62)
  - Target ' ': prob: 0.0000, logit: 14.31
  - Target '2': prob: 0.0003, logit: 22.38
  - Target '3': prob: 0.0008, logit: 23.50
  - Target '4': prob: 1.0000, logit: 30.62
  - Target '7

## Supporting diagnostic - SAE feature swap (weak directional effect)

Swap features from a **no-carry** source into the **carry** target. This is a coarse
activation-transfer probe, not a clean carry-isolation test, because the source is a
two-digit answer while the target is the tens digit of a three-digit answer:

- **Target (carry):** `58 + 83`, prompt `... Answer: 1`. Correct tens digit is `4`
  (5+8+**1**=14). If the carry is dropped, the tens digit becomes `3` (5+8=13).
- **Source (no-carry):** `32 + 42`, prompt `... Answer: ` - its next token is `7`
  (the tens digit of `74`), with no incoming carry.

We track `4` (carry kept, correct), `3` (the dropped-carry alternative), and `7`
(the source's own next digit). Because Qwen tokenizes
digits individually and `4` / `3` are now **distinct single tokens**, this contrast
is finally measurable - the previous `141` vs `131` targets both collapsed to the
first token `1` and were indistinguishable.

**Caveat:** because the source's own next token is `7`, this swap is more likely to
show source-digit leakage toward `7` than a clean shift toward `3`. A clean `3`
contrast would need a source prompt whose next token is actually `3`.


In [15]:
# SAE feature swap: 32+42 source -> 58+83 target at the next-token position
# Track 4 (carry kept), 3 (dropped-carry alternative), and 7 (source's own next digit).
!python src/intervention.py \
  --mode swap \
  --source-prompt "Question: What is 32 + 42? Answer: " \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --layers 4 8 12 16 20 24 28 \
  --graph-json outputs/add_58_83_t_graph.json \
  --graph-feature-sign positive \
  --target-token "4, 3, 7" \
  --output outputs/add_swap_nocarry_to_carry.json

Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 177.57it/s]
Loaded 157 nodes from graph JSON; extracted features across 7 layers (70 total features)
  Graph feature sign filter: positive (skipped 70 feature nodes)

[1/3] Clean Model Baseline:
  - Top prediction: '4' (prob: 1.0000, logit: 30.62)
  - Target ' ': prob: 0.0000, logit: 14.31
  - Target '1': prob: 0.0001, logit: 21.75
  - Target '2': prob: 0.0003, logit: 22.38
  - Target '3': prob: 0.0008, logit: 23.50
  - Target '4': prob: 1.0000, logit: 30.62
  - Target '7': prob: 0.0000, logit: 19.50

Source baseline prediction on 'Question: What is 32 + 42? Answer: ':
  - Target ' ': prob: 0.0000, logit: 14.12
  - Target '1': prob: 0.0026, logit: 22.62
  - Target '2': prob: 0.0011, logit: 21.75
  - Target '3': prob: 0.0527, logit: 25.62
  - Target '4': prob: 0.0016, logit: 22.12
  - Target '7': prob: 0.9375,

---
## Copy all outputs to Drive

In [17]:
# Copy all addition outputs to Google Drive for persistence
import os, shutil, glob
drive_out = '/content/drive/MyDrive/mphil-project/outputs'
os.makedirs(drive_out, exist_ok=True)
for src in glob.glob('/content/outputs/add_*'):
    if os.path.isfile(src):
        shutil.copy2(src, drive_out)
        print('Copied', os.path.basename(src))
print('Done!')

Copied add_58_83_h_graph.json
Copied add_58_83_h_graph.html
Copied add_swap_full_nocarry_to_carry.json
Copied add_knockout_attn_58_83.json
Copied add_58_83_u_graph.json
Copied add_58_83_u_graph.html
Copied add_knockout_block_58_83.json
Copied add_swap_nocarry_to_carry.json
Copied add_knockout_58_83.json
Copied add_58_83_u_graph.md
Copied add_scan_58_83.json
Copied add_58_83_t_graph.html
Copied add_58_83_t_graph.md
Copied add_58_83_h_graph.md
Copied add_58_83_t_graph.json
Copied add_inhibit_58_83.json
Done!


---
## Notes on interpreting these results

- **Per-digit is necessary, not optional.** Qwen emits numbers one digit at a time, most-significant first. A whole-answer graph only attributes to the first digit.
- **Use the contrast graph for carry.** The key tens graph should be built with `--target "4" --contrast-target "3"`, so interventions target the preference for the correct carry digit over the dropped-carry alternative.
- **MLP knockout negative result:** zeroing final-token MLP outputs did not reduce `4`, so the carry answer is not localized in the hooked MLP outputs alone.
- **Attention knockout partial result:** attention knockout reducing the `4` logit is useful evidence that attention/residual pathways contribute to the next digit. Use `--layer-scan` to identify which layers carry the effect.
- **Block knockout is an upper bound.** It destroys the residual stream at the final token, so a catastrophic result is expected and not a clean circuit localization. It tells you these layers are necessary in aggregate, not which SAE features implement carry.
- **The 32+42 swap is a coarse probe.** Its source next token is `7`, not `3`, so it is useful for source-leakage/no-carry transfer but not a clean `141 -> 131` intervention.
- **Layer 32 SAE remains suspect.** Keep using `--layers 4 8 12 16 20 24 28` unless you explicitly want to report the degenerate layer-32 SAE as a limitation.


---
## Main result - All-position MLP knockout

This is the strongest causal result in the arithmetic notebook. Final-token-only MLP knockout did not weaken the correct digit, but all-position MLP knockout across layers 4,8,12,16,20,24,28 collapsed the `4` vs `3` margin from 7.12 to 1.25:

- clean: logit(`4`) = 30.62, logit(`3`) = 23.50
- all-position MLP knockout: logit(`4`) = 27.25, logit(`3`) = 26.00
- `P(4)` fell to 0.6406; `P(3)` rose to 0.1836

Interpret this as evidence that carry-relevant arithmetic information is distributed across prompt positions in MLP/residual updates. It is not a clean sparse SAE carry circuit, because `2` also rises strongly.

Run the token-map cell first, then this all-position MLP knockout. The all-position attention and all-position SAE-feature inhibition cells below are supporting negative/limitation checks.


In [28]:
# Print token positions for choosing non-final intervention sites
!python src/intervention.py \
  --mode inhibit \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --print-tokens

Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 186.87it/s]
Prompt token positions:
   0: id=14582    token='Question'
   1: id=25       token=':'
   2: id=3555     token=' What'
   3: id=374      token=' is'
   4: id=220      token=' '
   5: id=20       token='5'
   6: id=23       token='8'
   7: id=488      token=' +'
   8: id=220      token=' '
   9: id=23       token='8'
  10: id=18       token='3'
  11: id=30       token='?'
  12: id=21806    token=' Answer'
  13: id=25       token=':'
  14: id=220      token=' '
  15: id=16       token='1'
Use --positions last, --positions all, or comma-separated indices such as --positions 4,5,9


In [29]:
# All-position MLP knockout: tests whether the negative MLP result was final-token-only
!python src/intervention.py \
  --mode inhibit \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --target-token "4, 3, 7" \
  --layers 4 8 12 16 20 24 28 \
  --full-knockout \
  --knockout-component mlp \
  --layer-scan \
  --positions all \
  --output outputs/add_knockout_mlp_allpos_58_83.json

Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 187.66it/s]

[1/3] Clean Model Baseline:
  - Top prediction: '4' (prob: 1.0000, logit: 30.62)
  - Target ' ': prob: 0.0000, logit: 14.31
  - Target '2': prob: 0.0003, logit: 22.38
  - Target '3': prob: 0.0008, logit: 23.50
  - Target '4': prob: 1.0000, logit: 30.62
  - Target '7': prob: 0.0000, logit: 19.50

[2/3] Running SAE Reconstruction-only Baseline (no features inhibited)...
Running forward pass with inhibition at positions [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]: {}...
Editing token positions:
   0: id=14582    token='Question'
   1: id=25       token=':'
   2: id=3555     token=' What'
   3: id=374      token=' is'
   4: id=220      token=' '
   5: id=20       token='5'
   6: id=23       token='8'
   7: id=488      token=' +'
   8: id=220      token=' '
   9: id=23       token='8'
  10:

In [45]:
# All-position attention knockout: tests whether attention effects are stronger away from final token
!python src/intervention.py \
  --mode inhibit \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --target-token "4, 3, 7" \
  --layers 4 8 12 16 20 24 28 \
  --full-knockout \
  --knockout-component attn \
  --positions all \
  --output outputs/add_knockout_attn_allpos_58_83.json

Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 184.29it/s]

[1/3] Clean Model Baseline:
  - Top prediction: '4' (prob: 1.0000, logit: 30.62)
  - Target ' ': prob: 0.0000, logit: 14.31
  - Target '2': prob: 0.0003, logit: 22.38
  - Target '3': prob: 0.0008, logit: 23.50
  - Target '4': prob: 1.0000, logit: 30.62
  - Target '7': prob: 0.0000, logit: 19.50

[2/3] Running SAE Reconstruction-only Baseline (no features inhibited)...
Running forward pass with inhibition at positions [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]: {}...
Editing token positions:
   0: id=14582    token='Question'
   1: id=25       token=':'
   2: id=3555     token=' What'
   3: id=374      token=' is'
   4: id=220      token=' '
   5: id=20       token='5'
   6: id=23       token='8'
   7: id=488      token=' +'
   8: id=220      token=' '
   9: id=23       token='8'
  10:

In [43]:
# All-position graph-positive SAE-feature inhibition on the 4-vs-3 contrast graph
# This is the closest check for whether SAE carry features exist away from the final token.
!python src/intervention.py \
  --mode inhibit \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --target-token "4, 3, 7" \
  --layers 4 8 12 16 20 24 28 \
  --graph-json outputs/add_58_83_t_graph.json \
  --graph-feature-sign positive \
  --positions all \
  --output outputs/add_inhibit_allpos_58_83.json

Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 185.24it/s]
Loaded 157 nodes from graph JSON; extracted features across 7 layers (70 total features)
  Graph feature sign filter: positive (skipped 70 feature nodes)

[1/3] Clean Model Baseline:
  - Top prediction: '4' (prob: 1.0000, logit: 30.62)
  - Target ' ': prob: 0.0000, logit: 14.31
  - Target '2': prob: 0.0003, logit: 22.38
  - Target '3': prob: 0.0008, logit: 23.50
  - Target '4': prob: 1.0000, logit: 30.62
  - Target '7': prob: 0.0000, logit: 19.50

[2/3] Running SAE Reconstruction-only Baseline (no features inhibited)...
Running forward pass with inhibition at positions [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]: {}...
Editing token positions:
   0: id=14582    token='Question'
   1: id=25       token=':'
   2: id=3555     token=' What'
   3: id=374      token=' is'
   4: id=220      to

In [ ]:
# Persist non-final-position diagnostics
import os, shutil, glob
drive_out = '/content/drive/MyDrive/mphil-project/outputs'
os.makedirs(drive_out, exist_ok=True)
for pattern in ('/content/outputs/add_*allpos*',):
    for src in glob.glob(pattern):
        if os.path.isfile(src):
            shutil.copy2(src, drive_out)
            print('Copied', os.path.basename(src))
print('Done!')
